# 🎨 Universal Image LoRA Trainer (Google Colab)
Bộ công cụ huấn luyện LoRA Hình Ảnh đa năng, tối ưu hóa bộ nhớ VRAM cho **FLUX.1, FLUX.2 Klein, Qwen-Image, Qwen-Image-Edit, Z-Image Turbo và Krea2**.

---
### 📚 Hướng Dẫn Chuẩn Bị Dữ Liệu Cho Từng Dạng LoRA:
1. **LoRA Nhân vật / Concept / Phong cách (Standard LoRA)**:
   - Đặt toàn bộ ảnh và file `.txt` caption vào một thư mục (VD: `/content/drive/MyDrive/TranningLorasData/20_model_girl`).
   - Có thể đặt tên thư mục theo cú pháp `{repeats}_{tên}` để tự nhận số lần lặp.
2. **LoRA Xử lý Da / Retouch / Phục hồi / Upscale (Paired Control LoRA)**:
   - Chuẩn bị 2 thư mục:
     - `Control_Folder`: Chứa ảnh đầu vào (ảnh da mụn/thô, ảnh mờ/nén).
     - `Train_Folders`: Chứa ảnh kết quả chất lượng cao (ảnh da đã chỉnh đẹp, ảnh gốc 4K siêu nét).
     - **Lưu ý**: Ảnh ở 2 thư mục phải có **tên tệp giống nhau 1-1** (VD: `001.jpg`, `002.jpg`).

### ☕ Bước 1: Khởi tạo Môi trường & Nhận diện GPU

In [ ]:
# @title ⚙️ 1. Cài đặt Thư viện & Kiểm tra GPU & Key Vault
# @markdown Nhấn nút Play để cài đặt môi trường và kiểm tra các API Key đã lưu trên Google Drive:

import os
import sys

# Mount Google Drive
from google.colab import drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# Cài đặt các gói phụ thuộc
!pip install -q toml pyyaml python-dotenv bitsandbytes optimum-quanto google-genai openai accelerate safetensors huggingface_hub tqdm pillow ipywidgets voluptuous imagesize einops ftfy regex sentencepiece protobuf scipy wandb lion-pytorch prodigyopt albumentations

# Clone / Cập nhật repo chính thức
if not os.path.exists('/content/TranningLoras'):
    !git clone https://github.com/nguyenducvuongg/TranningLoras.git /content/TranningLoras
else:
    !git -C /content/TranningLoras pull

%cd /content/TranningLoras
!pip install -q -e .

from lora_trainer.engine.hardware import detect_hardware_environment, setup_cuda_environment
from lora_trainer.caption.key_manager import display_key_vault_dashboard
hw_info = detect_hardware_environment()

print("\n" + "="*60)
print(f"🚀 GPU: {hw_info['gpu_name']} | VRAM: {hw_info['vram_gb']} GB ({hw_info['device_tier']})")
print(f"⚡ Khuyến nghị: FP8 = {hw_info['recommended_fp8']} | Batch Size = {hw_info['recommended_batch_size']} | Res = {hw_info['recommended_resolution']}")
print("="*60 + "\n")

# Hiển thị bảng điều khiển API Key Vault đã lưu từ các phiên trước
display_key_vault_dashboard()

### 🔐 (Tùy chọn) Quản lý & Thêm API Key vào Vault

In [ ]:
# @title 🔐 Quản lý API Key Vault (Thêm / Đổi Key)
# @markdown Dùng form này nếu bạn muốn lưu trước API Key mới vào Google Drive:
Platform = "gemini" # @param ["gemini", "huggingface", "wandb", "openai", "civitai"]
New_API_Key = "" # @param {type:'string'}
Key_Label = "" # @param {type:'string'}

from lora_trainer.caption.key_manager import save_api_key, display_key_vault_dashboard
if New_API_Key.strip():
    save_api_key(Platform, New_API_Key, label=Key_Label or None, set_default=True)
display_key_vault_dashboard()

### 📂 Bước 2: Chuẩn bị Dữ liệu & AI Captioning Chuyên Sâu

In [ ]:
# @title 📂 2. Cấu hình Dữ liệu & AI Captioning
# @markdown Nhập đường dẫn thư mục ảnh trên Google Drive:

Train_Folders = "/content/drive/MyDrive/LoRA_Data/MyConcept" # @param {type:'string'}
# @markdown 💡 `Control_Folder chỉ dùng khi train LoRA dạng Edit / Inpainting / Xử lý da / Upscale (như FLUX Kontext, Qwen Edit)`
Control_Folder = "" # @param {type:'string'}
Clean_Data = True # @param {type:'boolean'}

# @markdown 🤖 **Tùy chọn AI Captioning**
Caption_Engine = "Gemini-3.6-Flash" # @param ["None", "Gemini-3.6-Flash", "Gemini-3.7-Flash", "Gemini-3.5-Flash", "Gemini-3.5-Flash-Lite", "Gemini-3.1-Pro", "Gemini-3-Pro", "Florence-2", "JoyCaption", "OpenAI-GPT4o"]
# @markdown 🎯 **Chế độ Prompt chuyên biệt theo mục đích LoRA:**
Task_Mode = "General" # @param ["General", "Skin_Portrait", "Upscale_Restoration", "Art_Style", "Character_Outfit"]
Caption_Length = "Medium" # @param ["Short", "Medium", "Long"]

# @markdown 🔑 **API Key (Tự động ghi nhớ)**: Để trống nếu muốn dùng Key đã lưu trước đó trong Vault.
API_Key = "" # @param {type:'string'}
Custom_Trigger_Word = "" # @param {type:'string'}
Add_Folder_Name = False # @param {type:'boolean'}
Overwrite_Existing_Captions = False # @param {type:'boolean'}

from lora_trainer.data.cleaner import clean_directory
from lora_trainer.data.dataset_builder import build_dataset_list, check_folder_stats
from lora_trainer.data.tag_processor import process_dir_tags, add_folder_name_tags
from lora_trainer.caption.gemini_captioner import batch_caption_gemini
from lora_trainer.caption.florence_captioner import batch_caption_florence
from lora_trainer.caption.joy_captioner import batch_caption_joycaption
from lora_trainer.caption.openai_captioner import batch_caption_openai

folder_list = [f.strip() for f in Train_Folders.split(",") if f.strip()]

for folder in folder_list:
    if Clean_Data:
        rm_count, valid_count = clean_directory(folder)
        print(f"🧹 Đã dọn dẹp {rm_count} tệp rác. Còn lại {valid_count} tệp hợp lệ trong {folder}")

    # Chạy Captioning (tự động nhận diện và lưu API Key vào Vault)
    if Caption_Engine.startswith("Gemini"):
        batch_caption_gemini(
            folder,
            api_key=API_Key,
            model_alias=Caption_Engine,
            length_preset=Caption_Length,
            task_mode=Task_Mode,
            overwrite=Overwrite_Existing_Captions,
        )
    elif Caption_Engine == "Florence-2":
        batch_caption_florence(folder, task_preset=Caption_Length, overwrite=Overwrite_Existing_Captions)
    elif Caption_Engine == "JoyCaption":
        batch_caption_joycaption(folder, caption_length=Caption_Length.lower(), overwrite=Overwrite_Existing_Captions)
    elif Caption_Engine == "OpenAI-GPT4o":
        batch_caption_openai(folder, api_key=API_Key, length_preset=Caption_Length, overwrite=Overwrite_Existing_Captions)

    if Add_Folder_Name:
        add_folder_name_tags(folder)

    if Custom_Trigger_Word:
        process_dir_tags(folder, Custom_Trigger_Word, append=False)

    stats = check_folder_stats(folder)
    print(f"📊 Thống kê {folder}: {stats.get('total_images', 0)} ảnh, {stats.get('captioned_files', 0)} caption ({stats.get('caption_ratio_pct', 0)}%)")

### 🚀 Bước 3: Cấu hình Model & Khởi chạy Huấn luyện
💡 **Gợi ý thiết lập cho các bài toán đặc biệt:**
- **LoRA Xử lý Da / Portrait**: Chọn `FLUX.1-Kontext-dev` hoặc `Qwen-Image-Edit`, nhập `Control_Folder`, đặt LR = 1e-4, Dim = 32, Alpha = 16.
- **LoRA Upscale / Tăng nét**: Chọn `FLUX.1-Kontext-dev`, đặt Dim = 16, Alpha = 16, LR = 1.5e-4.
- **LoRA Phong cách (Art Style)**: Chọn `FLUX.2-klein-base-9B` hoặc `Z-Image-Turbo`, không nhập Control_Folder, đặt LR = 1e-4, Dim = 32.

In [ ]:
# @title 🛠️ 3. Thiết lập Tham số & Bắt đầu Huấn luyện
# @markdown 📂 **Thư mục Dữ liệu**:
Train_Folders = "/content/drive/MyDrive/LoRA_Data/MyConcept" # @param {type:'string'}
Control_Folder = "" # @param {type:'string'}

Model_Type = "FLUX.2-klein-base-9B" # @param ["FLUX.2-klein-base-9B", "FLUX.2-klein-base-4B", "Qwen-Image", "Qwen-Image-Edit", "Qwen-Image-Edit-2509", "Qwen-Image-Edit-2511", "Z-Image-Turbo", "Z-Image-Base", "Z-Image-De-Turbo", "FLUX.1-Kontext-dev", "FLUX.1-dev", "FLUX.1-schnell", "Krea2-Raw"]

Output_Directory = "/content/drive/MyDrive/LoRA_Outputs" # @param {type:'string'}
LoRA_Name = "my_awesome_lora" # @param {type:'string'}

Resolution = "1024,1024" # @param {type:'string'}
Batch_Size = 1 # @param {type:'integer'}
Learning_Rate = 1e-4 # @param {type:'number'}
Optimizer = "adamw8bit" # @param ["adamw8bit", "adamw", "adafactor"]
LR_Scheduler = "constant" # @param ["constant", "cosine", "linear", "polynomial"]
Network_Dim = 32 # @param {type:'integer'}
Network_Alpha = 16 # @param {type:'integer'}

Max_Train_Epochs = 10 # @param {type:'integer'}
Save_Every_N_Epochs = 2 # @param {type:'integer'}
Sample_Every_N_Steps = 200 # @param {type:'integer'}
Sample_Prompt = "" # @param {type:'string'}

# @markdown 📊 **Tùy chọn Bổ sung (Tự động ghi nhớ vào Vault)**:
# @markdown 💡 *Để trống các ô dưới nếu bạn muốn dùng Token/Key đã lưu từ trước hoặc Mirror công khai.*
HF_Token = "" # @param {type:'string'}
WandB_API_Key = "" # @param {type:'string'}
Auto_Disconnect = False # @param {type:'boolean'}

import os
from lora_trainer.config.model_registry import get_model_info, get_preferred_engine
from lora_trainer.config.musubi_config import MusubiConfigBuilder
from lora_trainer.config.toolkit_config import ToolkitConfigBuilder
from lora_trainer.engine.downloader import download_model_suite
from lora_trainer.engine.musubi_runner import run_musubi_pipeline
from lora_trainer.engine.toolkit_runner import run_toolkit_pipeline
from lora_trainer.data.dataset_builder import build_dataset_list
from lora_trainer.utils.sampler import get_random_sample_prompt
from lora_trainer.utils.converter import auto_convert_checkpoints
from lora_trainer.utils.colab_utils import auto_disconnect

res_list = [int(x.strip()) for x in Resolution.split(",") if x.strip()]
if len(res_list) == 1:
    res_list = [res_list[0], res_list[0]]

datasets = build_dataset_list(Train_Folders, Control_Folder)
engine_type = get_preferred_engine(Model_Type)
print(f"🎯 Mô hình: {Model_Type} | Engine tối ưu: {engine_type.upper()}")

# Tải trước các trọng số cần thiết (tự động hỗ trợ HF Token từ Vault hoặc Mirror công khai)
weights = download_model_suite(Model_Type, weights_dir="/content/models", hf_token=HF_Token)

if engine_type == "musubi":
    builder = MusubiConfigBuilder(
        model_name=Model_Type,
        output_dir=Output_Directory,
        output_name=LoRA_Name,
        cache_base_dir="/content/cache",
        weights_dir="/content/models",
    )
    
    dataset_toml = "/content/dataset.toml"
    builder.build_dataset_toml(
        dataset_path=dataset_toml,
        resolution=res_list,
        image_folders=datasets,
    )

    vae_path = weights.get("vae", "")
    clip1_path = weights.get("text_encoder1", "")
    clip2_path = weights.get("text_encoder2", None)
    clip_vision = weights.get("clip_vision", None)
    dit_path = weights.get("dit", "")

    cache_latents_cmd = builder.build_cache_latents_args(dataset_toml, vae_path, clip_vision) if vae_path else None
    cache_te_cmd = builder.build_cache_text_encoder_args(dataset_toml, clip1_path, clip2_path) if clip1_path else None

    sample_txt_path = "/content/sample_prompt.txt"
    if Sample_Prompt == "":
        p, img, ctrl = get_random_sample_prompt(datasets[0]["path"], datasets[0].get("control_path"))
        Sample_Prompt = p
    with open(sample_txt_path, "w", encoding="utf-8") as f:
        f.write(f"{Sample_Prompt} --w {res_list[0]} --h {res_list[1]}\n")

    train_cmd = builder.build_train_args(
        dataset_config_path=dataset_toml,
        dit_model_path=dit_path,
        learning_rate=Learning_Rate,
        optimizer_type=Optimizer,
        lr_scheduler=LR_Scheduler,
        network_dim=Network_Dim,
        network_alpha=Network_Alpha,
        max_train_epochs=Max_Train_Epochs,
        save_every_n_epochs=Save_Every_N_Epochs,
        sample_prompt_file=sample_txt_path if Sample_Every_N_Steps > 0 else None,
        sample_every_n_steps=Sample_Every_N_Steps,
        wandb_api_key=WandB_API_Key if WandB_API_Key else None,
    )

    run_musubi_pipeline(
        musubi_dir="/content/musubi-tuner",
        cache_latents_cmd=cache_latents_cmd,
        cache_text_encoder_cmd=cache_te_cmd,
        train_cmd=train_cmd,
    )

else:
    builder = ToolkitConfigBuilder(
        model_name=Model_Type,
        output_dir=Output_Directory,
        output_name=LoRA_Name,
    )
    yaml_path = "/content/toolkit_config.yaml"
    builder.build_yaml_config(
        save_yaml_path=yaml_path,
        dataset_folders=datasets,
        steps=Max_Train_Epochs * 200,
        save_every=Save_Every_N_Epochs * 200,
        batch_size=Batch_Size,
        learning_rate=Learning_Rate,
        linear_dim=Network_Dim,
        linear_alpha=Network_Alpha,
        sample_prompts=[Sample_Prompt] if Sample_Prompt else None,
        sample_every=Sample_Every_N_Steps,
        sample_resolution=res_list,
        wandb_api_key=WandB_API_Key if WandB_API_Key else None,
    )
    run_toolkit_pipeline(config_yaml_path=yaml_path, toolkit_dir="/content/ai-toolkit")

# ⚡ TỰ ĐỘNG NHẬN DIỆN VÀ CHUYỂN ĐỔI SANG COMFYUI NẾU MÔ HÌNH YÊU CẦU (Ví dụ Z-Image)
auto_convert_checkpoints(Output_Directory, Model_Type)

if Auto_Disconnect:
    auto_disconnect(delay_seconds=120, enabled=True)

### 🛠️ Bước 4: Công Cụ Chuyển Đổi Thủ Công LoRA Sang ComfyUI (Tùy Chọn)
💡 **Lưu ý**: Hệ thống ở Bước 3 đã **tự động phát hiện và chuyển đổi** các file LoRA (như Z-Image) sang định dạng ComfyUI trong thư mục `ComfyUI_Ready`.
Nếu bạn muốn tự chuyển đổi thủ công một file bất kỳ:
- **`Input_LoRA`**: Đường dẫn file `.safetensors` gốc sau khi train (ví dụ: `/content/drive/MyDrive/LoRA_Outputs/my_awesome_lora-000010.safetensors`).
- **`Output_LoRA`**: Đường dẫn file `.safetensors` mới sẵn sàng cho ComfyUI (ví dụ: `/content/drive/MyDrive/LoRA_Outputs/comfy_my_awesome_lora-000010.safetensors`).

In [ ]:
# @title 🔄 Convert Z-LoRA to ComfyUI (Manual Tool)
Input_LoRA = "/content/drive/MyDrive/LoRA_Outputs/my_awesome_lora-000010.safetensors" # @param {type:'string'}
Output_LoRA = "/content/drive/MyDrive/LoRA_Outputs/comfy_my_awesome_lora.safetensors" # @param {type:'string'}

from lora_trainer.utils.converter import convert_z_lora_to_comfyui
if Input_LoRA and Output_LoRA and os.path.exists(Input_LoRA):
    convert_z_lora_to_comfyui(Input_LoRA, Output_LoRA)
else:
    print("⚠️ Vui lòng kiểm tra lại đường dẫn file Input_LoRA!")